In [5]:
import os
from pathlib import Path
from dotenv import load_dotenv
import mlflow

load_dotenv(Path(".") / ".env")  # your .env is inside notebooks/

mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI"))
mlflow.set_experiment("cosmetics_experiments_rubricB")

print("✅ MLflow connected:", mlflow.get_tracking_uri())


✅ MLflow connected: http://dagshub.com/rvaghani/my-first-repo.mlflow


In [ ]:
import pandas as pd

df = pd.read_sql("""
SELECT
    review_text,
    rating,
    helpful_votes,
    verified_purchase,
    label
FROM review
""", engine)

df["verified_purchase"] = df["verified_purchase"].astype(int)
df["helpful_votes"] = df["helpful_votes"].fillna(0).astype(int)
df["rating"] = df["rating"].astype(float)
df["label"] = df["label"].astype(int)
df["low_rating"] = (df["rating"] <= 2).astype(int)


X = df[["review_text", "rating", "helpful_votes", "verified_purchase"]]
y = df["label"]

print(df.shape)
print(y.value_counts(normalize=True))


(49877, 5)
label
1    0.740842
0    0.259158
Name: proportion, dtype: float64


In [12]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

TEXT_COL = "review_text"
NUM_COLS = ["rating", "helpful_votes"]
CAT_COLS = ["verified_purchase"]  # for OneHot

log1p = FunctionTransformer(lambda x: np.log1p(x), validate=False)

def make_pipeline(scaler="standard", C=1.0):
    if scaler == "standard":
        num_scaler = StandardScaler(with_mean=False)  # safe with sparse
    elif scaler == "minmax":
        num_scaler = MinMaxScaler()
    else:
        raise ValueError("scaler must be 'standard' or 'minmax'")

    num_pipe = Pipeline([
        ("log", log1p),
        ("scaler", num_scaler)
    ])

    pre = ColumnTransformer(
        transformers=[
            ("text", TfidfVectorizer(max_features=5000, stop_words="english"), TEXT_COL),
            ("num", num_pipe, NUM_COLS),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS)
        ]
    )

    clf = LogisticRegression(C=C, max_iter=2000, solver="liblinear")
    pipe = Pipeline([("pre", pre), ("clf", clf)])
    return pipe


In [16]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

def log_votes(arr):
    arr = arr.copy()
    arr[:, 1] = np.log1p(arr[:, 1])  # helpful_votes
    return arr

num_pipe = Pipeline([
    ("log_votes", FunctionTransformer(log_votes, validate=True)),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("text", TfidfVectorizer(max_features=20000, stop_words="english", ngram_range=(1,2)), "review_text"),
    ("num", num_pipe, ["rating", "helpful_votes"]),
    ("verified", "passthrough", ["verified_purchase"]),
], sparse_threshold=0.3)


In [13]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import f1_score, confusion_matrix

def run_exp1(run_name, scaler, folds, C=1.0):
    pipe = make_pipeline(scaler=scaler, C=C)
    cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)

    # CV f1 mean/std
    scores = cross_val_score(pipe, X, y, cv=cv, scoring="f1")
    f1_mean = float(scores.mean())
    f1_std = float(scores.std())

    # Confusion matrix counts from CV predictions
    preds = cross_val_predict(pipe, X, y, cv=cv)
    tn, fp, fn, tp = confusion_matrix(y, preds).ravel()

    # Train on full data and score (for rubric)
    pipe.fit(X, y)
    train_pred = pipe.predict(X)
    f1_train = float(f1_score(y, train_pred))

    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("experiment", "exp1")
        mlflow.log_param("model", "logreg")
        mlflow.log_param("scaler", scaler)
        mlflow.log_param("folds", folds)
        mlflow.log_param("C", C)

        mlflow.log_metric("f1_cv_mean", f1_mean)
        mlflow.log_metric("f1_cv_std", f1_std)
        mlflow.log_metric("f1_train", f1_train)

        mlflow.log_metric("tp", int(tp))
        mlflow.log_metric("tn", int(tn))
        mlflow.log_metric("fp", int(fp))
        mlflow.log_metric("fn", int(fn))

    print(f"✅ {run_name}: f1_cv_mean={f1_mean:.4f}, f1_cv_std={f1_std:.4f}, f1_train={f1_train:.4f}")
    print("TP TN FP FN:", tp, tn, fp, fn)
    return f1_mean


In [14]:
run_exp1("EXP1_logreg_standard_3fold", scaler="standard", folds=3, C=1.0)
run_exp1("EXP1_logreg_standard_10fold", scaler="standard", folds=10, C=1.0)

run_exp1("EXP1_logreg_minmax_3fold", scaler="minmax", folds=3, C=1.0)
run_exp1("EXP1_logreg_minmax_10fold", scaler="minmax", folds=10, C=1.0)


🏃 View run EXP1_logreg_standard_3fold at: http://dagshub.com/rvaghani/my-first-repo.mlflow/#/experiments/1/runs/ae3f9e2767294a32bb8997cd8994169d
🧪 View experiment at: http://dagshub.com/rvaghani/my-first-repo.mlflow/#/experiments/1
✅ EXP1_logreg_standard_3fold: f1_cv_mean=1.0000, f1_cv_std=0.0000, f1_train=1.0000
TP TN FP FN: 36951 12926 0 0
🏃 View run EXP1_logreg_standard_10fold at: http://dagshub.com/rvaghani/my-first-repo.mlflow/#/experiments/1/runs/2fc4944899484828b85b3dde6d17a536
🧪 View experiment at: http://dagshub.com/rvaghani/my-first-repo.mlflow/#/experiments/1
✅ EXP1_logreg_standard_10fold: f1_cv_mean=1.0000, f1_cv_std=0.0000, f1_train=1.0000
TP TN FP FN: 36951 12926 0 0
🏃 View run EXP1_logreg_minmax_3fold at: http://dagshub.com/rvaghani/my-first-repo.mlflow/#/experiments/1/runs/a2dcadcf79fb44cabe71e7ef72cdc4c8
🧪 View experiment at: http://dagshub.com/rvaghani/my-first-repo.mlflow/#/experiments/1
✅ EXP1_logreg_minmax_3fold: f1_cv_mean=1.0000, f1_cv_std=0.0000, f1_train=1.0000

1.0

In [15]:
import optuna

def tune_logreg_C(scaler="standard", folds=3, n_trials=10):
    cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)

    def objective(trial):
        C = trial.suggest_float("C", 1e-3, 10.0, log=True)
        pipe = make_pipeline(scaler=scaler, C=C)
        scores = cross_val_score(pipe, X, y, cv=cv, scoring="f1")
        return float(scores.mean())

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)
    return study.best_params, float(study.best_value)

best_params, best_score = tune_logreg_C(scaler="standard", folds=3, n_trials=10)
best_params, best_score


/Users/ar/Desktop/Riddhi/ESDS_Fall 2025/Pro Data science /Final Project/housing_app_fall25/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2025-12-18 00:50:22,812] A new study created in memory with name: no-name-2a713c99-a3fa-4d88-b4d0-cd17a82a8d04
[I 2025-12-18 00:50:25,348] Trial 0 finished with value: 0.9517707329037205 and parameters: {'C': 0.015098058819377232}. Best is trial 0 with value: 0.9517707329037205.
[I 2025-12-18 00:50:27,629] Trial 1 finished with value: 0.941559158502539 and parameters: {'C': 0.0071776627895933426}. Best is trial 0 with value: 0.9517707329037205.
[I 2025-12-18 00:50:30,640] Trial 2 finished with value: 1.0 and parameters: {'C': 7.767433715522158}. Best is trial 2 with value: 1.0.
[I 2025-12-18 00:50:33,382] Trial 3 finished with value: 0.9392858479406421 and p

({'C': 7.767433715522158}, 1.0)